In [1]:
import sys, torch
print(sys.version)        
print(torch.__version__)  

3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
2.8.0+cu126


In [2]:
!pip install torch torchvision --quiet
!pip install torch-geometric --quiet
!pip install matplotlib seaborn scipy tqdm --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 30.9 MB/s eta 0:00:0000:01


In [3]:
import glob

# Find actual wheel locations
wheels = glob.glob('/kaggle/input/**/*.whl', recursive=True)
for w in wheels:
    print(w)

print(f"\nPython: {sys.version}")
print(f"Torch:  {torch.__version__}")

/kaggle/input/private-dataset/torch_spline_conv-1.2.2pt29cpu-cp312-cp312-linux_x86_64.whl
/kaggle/input/private-dataset/torch_sparse-0.6.18pt29cpu-cp312-cp312-linux_x86_64.whl
/kaggle/input/private-dataset/torch_cluster-1.6.3pt29cpu-cp312-cp312-linux_x86_64.whl
/kaggle/input/private-dataset/torch_scatter-2.1.2pt29cpu-cp312-cp312-linux_x86_64.whl
/kaggle/input/datasets/aaryaupi/cached-artificats/wheels/torch_sparse-0.6.18-cp312-cp312-linux_x86_64.whl
/kaggle/input/datasets/aaryaupi/cached-artificats/wheels/torch_spline_conv-1.2.2-cp312-cp312-linux_x86_64.whl
/kaggle/input/datasets/aaryaupi/cached-artificats/wheels/torch_scatter-2.1.2-cp312-cp312-linux_x86_64.whl
/kaggle/input/datasets/aaryaupi/cached-artificats/wheels/torch_cluster-1.6.3-cp312-cp312-linux_x86_64.whl

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Torch:  2.8.0+cu126


# Pytorch Cuda Wheels to access PyG scatter sparse 

In [4]:
import subprocess

# Detect PyTorch and CUDA versions
torch_version = torch.__version__.split('+')[0]  # "2.8.0"
cuda_version = torch.version.cuda.replace('.', '')  # "126" from "12.6"

print(f"PyTorch: {torch_version}, CUDA: {cuda_version}")

# Install PyG extensions for your exact versions
pyg_url = f"https://data.pyg.org/whl/torch-{torch_version}+cu{cuda_version}.html"

packages = [
    'torch-scatter',
    'torch-sparse', 
    'torch-cluster',
    'torch-spline-conv'
]

for pkg in packages:
    print(f"\nInstalling {pkg}...")
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg, 
         '-f', pyg_url, '--no-cache-dir', '-q'],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f"  ✓ {pkg} installed successfully")
    else:
        print(f"  ✗ {pkg} failed: {result.stderr[:300]}")

PyTorch: 2.8.0, CUDA: 126

Installing torch-scatter...
  ✓ torch-scatter installed successfully

Installing torch-sparse...
  ✓ torch-sparse installed successfully

Installing torch-cluster...
  ✓ torch-cluster installed successfully

Installing torch-spline-conv...
  ✓ torch-spline-conv installed successfully


In [ ]:
from torch_geometric.transforms import SamplePoints, NormalizeScale, KNNGraph
import torch_geometric.transforms as T
from torch_geometric.loader import DataLoader
from torch_geometric.data import Batch, Data
from torch_geometric.nn import global_max_pool
from torch_cluster import knn_graph
from torch.cuda.amp import autocast, GradScaler
 
import os, torch, numpy as np
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import trange, tqdm
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from scipy.spatial import cKDTree
 
# ─────────────────────────────────────────────────────
# DEVICE / HYPERPARAMS
# ─────────────────────────────────────────────────────
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CACHE_PATH = '/kaggle/input/datasets/aaryaupi/cached-artificats/modelnet40_final.pt'
MOTIF_CACHE = '/kaggle/input/datasets/aaryaupi/gin-latest-pt-file/modelnet40_with_motifs.pt'
K_NEIGHBORS = 20
NUM_POINTS   = 1024
BATCH_SIZE   = 12  # Optimized for P100
EPOCHS       = 100
LR           = 1e-3
NUM_CLASSES  = 40
USE_AUGMENTATION = False  # Set to True to enable augmentation
 
# ─────────────────────────────────────────────────────
# 1. LOAD CACHED DATA
# ─────────────────────────────────────────────────────
print("Loading cached ModelNet40...")
import torch_geometric.data.data
torch.serialization.add_safe_globals([
    torch_geometric.data.data.DataEdgeAttr,
])
 
# Try to load motif cache first, fallback to base cache
if os.path.exists(MOTIF_CACHE):
    print(f"  Found motif cache: {MOTIF_CACHE}")
    cache = torch.load(MOTIF_CACHE, map_location='cpu', weights_only=False, mmap=True)
    train_list = cache['train']
    test_list = cache['test']
    CLASSES = cache['classes']
    print(f"  ✓ Loaded {len(train_list)} train + {len(test_list)} test samples")
    print(f"  ✓ Features already include motifs (shape: {train_list[0].x.shape})")
    MOTIFS_PRECOMPUTED = True
elif os.path.exists(CACHE_PATH):
    print(f"  Found base cache: {CACHE_PATH}")
    cache = torch.load(CACHE_PATH, map_location='cpu', weights_only=False, mmap=True)
    train_list = cache['train']
    test_list = cache['test']
    CLASSES = cache['classes']
    print(f"  ✓ Loaded {len(train_list)} train + {len(test_list)} test samples")
    print(f"  ⚠ Motifs not precomputed, will compute now...")
    MOTIFS_PRECOMPUTED = False
else:
    raise FileNotFoundError(f"Cache not found at {CACHE_PATH} or {MOTIF_CACHE}")

Loading cached ModelNet40...
  Found base cache: /kaggle/input/datasets/aaryaupi/cached-artificats/modelnet40_final.pt
  ✓ Loaded 9843 train + 2468 test samples
  ⚠ Motifs not precomputed, will compute now...


In [6]:
def compute_triangle_counts(edge_index: torch.Tensor, num_nodes: int) -> torch.Tensor:
    """
    Count triangles per node using sparse A² trick
    
    WHY THIS WORKS:
      For adjacency matrix A, (A²)[i,j] counts paths of length 2
      from i to j (common neighbors).
      A triangle exists when edge (i,j) exists AND i,j share a neighbor:
          triangles(i) = sum_j  A[i,j] * (A²)[i,j]
      Each triangle counted twice → divide by 2
    
    Complexity: O(N·k²) via matmul, ~20ms per cloud at N=1024, k=20
    
    Args:
        edge_index: [2, E] directed edges
        num_nodes: int N
    
    Returns:
        [N, 1] normalized triangle count per node
    """
    N = num_nodes
    src = edge_index[0]
    dst = edge_index[1]
    
    # Build dense adjacency matrix A [N, N]
    A = torch.zeros(N, N, dtype=torch.float32)
    A[src, dst] = 1.0
    
    # A² = A @ A
    A2 = A @ A  # [N, N] - counts common neighbors
    
    # Triangle count per node
    counts = (A * A2).sum(dim=1)  # [N]
    counts = counts / 2.0  # Each triangle counted twice
    
    # Normalize to [0, 1]
    max_c = counts.max()
    if max_c > 0:
        counts = counts / max_c
    
    return counts.unsqueeze(1)  # [N, 1]
 
 
def add_motif_features(data_list: list, k: int = 20, desc: str = '') -> list:
    """
    Add triangle motif counts as node features
    
    For each Data:
      1. Build static KNN graph from positions
      2. Compute triangle counts
      3. Append as extra feature → x becomes [N, 7]
    
    Args:
        data_list: list of PyG Data objects
        k: number of neighbors
        desc: progress bar description
    
    Returns:
        data_list with motif features added
    """
    for data in tqdm(data_list, desc=f'Motif scores {desc}', leave=False):
        pos = data.pos.numpy()  # [N, 3]
        N = pos.shape[0]
        
        # Build geometric KNN using cKDTree
        tree = cKDTree(pos)
        _, nn_idx = tree.query(pos, k=k + 1)  # [N, k+1]
        nn_idx = nn_idx[:, 1:]  # Drop self → [N, k]
        
        src = np.repeat(np.arange(N), k)  # [N*k]
        dst = nn_idx.flatten()  # [N*k]
        
        edge_index_geo = torch.tensor(
            np.stack([src, dst], axis=0),
            dtype=torch.long
        )
        
        # Compute triangle counts
        tri_counts = compute_triangle_counts(edge_index_geo, N)  # [N, 1]
        
        # Append to node features: x was [N, 6], now [N, 7]
        data.x = torch.cat([data.x, tri_counts], dim=1)
        
        # Store geometric edge_index (for reference)
        data.edge_index = edge_index_geo
    
    return data_list
 
 
# ─────────────────────────────────────────────────────
# 3. FARTHEST POINT SAMPLING
# ─────────────────────────────────────────────────────
def farthest_point_sample_simple(points: torch.Tensor, n_samples: int) -> torch.Tensor:
    """
    Simple FPS - selects maximally spread out points
    
    Args:
        points: [N, 3] point cloud (any device)
        n_samples: number of points to sample
    
    Returns:
        indices: [n_samples] indices of sampled points
    """
    N = points.shape[0]
    device = points.device
    
    if n_samples >= N:
        return torch.arange(N, device=device)
    
    sampled_indices = []
    distances = torch.ones(N, device=device) * 1e10
    
    # Start with random point
    current_idx = torch.randint(0, N, (1,), device=device).item()
    
    for _ in range(n_samples):
        sampled_indices.append(current_idx)
        
        # Update distances
        current_point = points[current_idx]
        dist_to_current = torch.norm(points - current_point, dim=1)
        distances = torch.minimum(distances, dist_to_current)
        
        # Next point = farthest from all sampled
        current_idx = torch.argmax(distances).item()
    
    return torch.tensor(sampled_indices, dtype=torch.long, device=device)
 
 
# ─────────────────────────────────────────────────────
# 4. POINT CLOUD AUGMENTATION
# ─────────────────────────────────────────────────────
def augment_pointcloud(pos: torch.Tensor, norm: torch.Tensor) -> tuple:
    """
    Enhanced augmentation for 3D point clouds
    
    Augmentations:
      1. Y-axis rotation (objects can face any direction)
      2. Isotropic scaling (different distances/sizes)
      3. Gaussian jitter (sensor noise)
    
    WHY:
      - Y-rotation: ModelNet objects have random yaw, gravity is fixed
      - Scaling: Objects appear at different distances
      - Jitter: Simulates sensor noise
    
    Reference: Qi et al. 2017 (PointNet), Section 5
    
    Args:
        pos: [N, 3] positions (any device)
        norm: [N, 3] normals (any device)
    
    Returns:
        aug_pos: [N, 3] on CPU
        aug_norm: [N, 3] on CPU
    """
    pos = pos.detach().cpu().clone()
    norm = norm.detach().cpu().clone()
    
    # 1. Y-axis rotation
    angle_y = np.random.rand() * 2 * np.pi
    cos_y, sin_y = np.cos(angle_y), np.sin(angle_y)
    
    Ry = torch.tensor([
        [ cos_y, 0, sin_y],
        [     0, 1,     0],
        [-sin_y, 0, cos_y]
    ], dtype=torch.float32)
    
    pos = pos @ Ry.T
    norm = norm @ Ry.T  # Rotate normals too!
    
    # 2. Isotropic scaling
    scale = 0.95 + np.random.rand() * 0.10  # [0.95, 1.05]
    pos = pos * scale
    # Don't scale normals (should stay unit vectors)
    
    # 3. Gaussian jitter
    noise = torch.randn_like(pos) * 0.01
    noise = torch.clamp(noise, -0.02, 0.02)
    pos = pos + noise
    
    return pos, norm
 
 
def apply_augmentation(batch_data):
    """
    Apply augmentation to batch (CPU processing)
    
    Args:
        batch_data: PyG Batch object
    
    Returns:
        Augmented batch
    """
    batch_data = batch_data.cpu()
    data_list = batch_data.to_data_list()
    
    for data in data_list:
        # Extract components
        pos = data.pos
        norm = data.norm if hasattr(data, 'norm') else data.x[:, 3:6]
        
        # Augment
        aug_pos, aug_norm = augment_pointcloud(pos, norm)
        
        # Update data
        data.pos = aug_pos
        if hasattr(data, 'norm'):
            data.norm = aug_norm
        
        # Rebuild x = [aug_pos | aug_norm | motif] or [aug_pos | aug_norm]
        if data.x.shape[1] == 7:
            # Has motif scores
            motif = data.x[:, 6:7]
            data.x = torch.cat([aug_pos, aug_norm, motif], dim=1)
        else:
            # No motif scores
            data.x = torch.cat([aug_pos, aug_norm], dim=1)
    
    return Batch.from_data_list(data_list)

In [9]:
if not MOTIFS_PRECOMPUTED:
    print("\nComputing triangle motif scores (one-time cost)...")
    train_list = add_motif_features(train_list, k=K_NEIGHBORS, desc='train')
    test_list = add_motif_features(test_list, k=K_NEIGHBORS, desc='test')
    
    # Save for next time
    print("Saving motif cache...")
    torch.save({
        'train': train_list,
        'test': test_list,
        'classes': CLASSES
    }, 'modelnet40_with_motifs.pt')
    print(f"✓ Saved to modelnet40_with_motifs.pt")
 
 
# ─────────────────────────────────────────────────────
# 6. PRECOMPUTE K-NN GRAPHS
# ─────────────────────────────────────────────────────
print("\n🚀 Precomputing k-NN graphs (one-time cost)...")
 
def add_cached_graph(data, k=20):
    """Add precomputed edge_index to data object"""
    edge_index = knn_graph(data.pos, k=k, loop=False)
    data.edge_index = edge_index
    return data
 
train_list = [add_cached_graph(d, k=K_NEIGHBORS) for d in tqdm(train_list, desc='Train graphs')]
test_list = [add_cached_graph(d, k=K_NEIGHBORS) for d in tqdm(test_list, desc='Test graphs')]
 
print("✅ Graphs cached!")
 
 
# ─────────────────────────────────────────────────────
# 7. OPTIMIZED DATALOADERS
# ─────────────────────────────────────────────────────
train_loader = DataLoader(
    train_list,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)
 
test_loader = DataLoader(
    test_list,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)
 
print(f"\nDataLoaders ready:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Test batches: {len(test_loader)}")
 
 
# ─────────────────────────────────────────────────────
# 8. OPTIMIZED DGCNN MODEL
# ─────────────────────────────────────────────────────
from torch_geometric.nn import EdgeConv as PyGEdgeConv
 
# ============================================================
# BASELINE DGCNN (original architecture, no pruning)
# ============================================================

class BaselineDGCNN(nn.Module):
    """
    Standard DGCNN as per Wang et al. 2019.
    - 4 EdgeConv layers with output channels 64, 64, 128, 256
    - Dynamic graph construction (k‑NN in feature space)
    - Global max + avg pooling → classifier
    """
    def __init__(self, in_channels=6, num_classes=40, k=20, dropout=0.5, 
                 embed_dims=1024, small=False):
        super().__init__()
        self.k = k
        if small:
            # Smaller model: reduce channels to speed up training
            ch1, ch2, ch3, ch4 = 32, 32, 64, 128
            embed_dims = 512
        else:
            ch1, ch2, ch3, ch4 = 64, 64, 128, 256

        # EdgeConv layers (single linear + BN + LeakyReLU)
        self.conv1 = EdgeConv(
            nn.Sequential(
                nn.Linear(2 * in_channels, ch1, bias=False),
                nn.BatchNorm1d(ch1),
                nn.LeakyReLU(0.2)
            ),
            aggr='max'
        )
        self.conv2 = EdgeConv(
            nn.Sequential(
                nn.Linear(2 * ch1, ch2, bias=False),
                nn.BatchNorm1d(ch2),
                nn.LeakyReLU(0.2)
            ),
            aggr='max'
        )
        self.conv3 = EdgeConv(
            nn.Sequential(
                nn.Linear(2 * ch2, ch3, bias=False),
                nn.BatchNorm1d(ch3),
                nn.LeakyReLU(0.2)
            ),
            aggr='max'
        )
        self.conv4 = EdgeConv(
            nn.Sequential(
                nn.Linear(2 * ch3, ch4, bias=False),
                nn.BatchNorm1d(ch4),
                nn.LeakyReLU(0.2)
            ),
            aggr='max'
        )

        # Global aggregation
        total_channels = ch1 + ch2 + ch3 + ch4
        self.global_mlp = nn.Sequential(
            nn.Linear(total_channels, embed_dims, bias=False),
            nn.BatchNorm1d(embed_dims),
            nn.LeakyReLU(0.2),
            nn.Dropout(dropout)
        )

        # Classifier
        if small:
            self.classifier = nn.Sequential(
                nn.Linear(embed_dims, 256, bias=False),
                nn.BatchNorm1d(256),
                nn.LeakyReLU(0.2),
                nn.Dropout(dropout),
                nn.Linear(256, 128, bias=False),
                nn.BatchNorm1d(128),
                nn.LeakyReLU(0.2),
                nn.Dropout(dropout),
                nn.Linear(128, num_classes)
            )
        else:
            self.classifier = nn.Sequential(
                nn.Linear(embed_dims, 512, bias=False),
                nn.BatchNorm1d(512),
                nn.LeakyReLU(0.2),
                nn.Dropout(dropout),
                nn.Linear(512, 256, bias=False),
                nn.BatchNorm1d(256),
                nn.LeakyReLU(0.2),
                nn.Dropout(dropout),
                nn.Linear(256, num_classes)
            )
    def forward(self, data):
        x = data.x[:, :6]          # pos + norm only
        batch = data.batch
        pos = data.pos

        # Dynamic graph at each layer
        edge_index = knn_graph(pos, k=self.k, batch=batch, loop=False)
        x1 = self.conv1(x, edge_index)

        edge_index = knn_graph(x1, k=self.k, batch=batch, loop=False)
        x2 = self.conv2(x1, edge_index)

        edge_index = knn_graph(x2, k=self.k, batch=batch, loop=False)
        x3 = self.conv3(x2, edge_index)

        edge_index = knn_graph(x3, k=self.k, batch=batch, loop=False)
        x4 = self.conv4(x3, edge_index)

        # Concatenate multi-scale features
        x = torch.cat([x1, x2, x3, x4], dim=1)

        # Global max + average pooling
        x_max = global_max_pool(x, batch)
        x_avg = global_mean_pool(x, batch)
        x = torch.cat([x_max, x_avg], dim=1)

        # Classifier
        x = self.global_mlp(x)
        x = self.classifier(x)
        return x
 


Computing triangle motif scores (one-time cost)...


Motif scores train:   0%|          | 0/9843 [00:00<?, ?it/s]

Motif scores test:   0%|          | 0/2468 [00:00<?, ?it/s]

Saving motif cache...
✓ Saved to modelnet40_with_motifs.pt

🚀 Precomputing k-NN graphs (one-time cost)...


Train graphs:   0%|          | 0/9843 [00:00<?, ?it/s]

Test graphs:   0%|          | 0/2468 [00:00<?, ?it/s]

✅ Graphs cached!

DataLoaders ready:
  Train batches: 1231
  Test batches: 309


In [ ]:
# ============================================================
# IMPORTS (ensure all needed are present)
# ============================================================
import os
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
from torch_geometric.data import Batch
from torch_geometric.nn import global_max_pool, global_mean_pool, EdgeConv
from torch_cluster import knn_graph
from tqdm.auto import trange, tqdm
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

# ============================================================
# CONFIGURATION (make sure these match your earlier settings)
# ============================================================
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
K_NEIGHBORS = 20
BATCH_SIZE = 8
EPOCHS = 150
LR = 5e-4
NUM_CLASSES = 40

# Paths
MOTIF_CACHE = '/kaggle/working/modelnet40_motif.pt'

# ============================================================
# LOAD DATA (with motif features, if available)
# ============================================================
print("Loading data...")
if os.path.exists(MOTIF_CACHE):
    print("Loading motif-augmented cache …")
    motif_cache = torch.load(MOTIF_CACHE, map_location='cpu', weights_only=False)
    train_list = motif_cache['train']
    test_list = motif_cache['test']
    CLASSES = motif_cache['classes']
else:
    print("Computing triangle motif scores (one-time cost) …")
    # The functions add_motif_features and compute_triangle_counts must be defined earlier
    train_list = add_motif_features(train_list, k=K_NEIGHBORS, desc='train')
    test_list = add_motif_features(test_list, k=K_NEIGHBORS, desc='test')
    torch.save({'train': train_list, 'test': test_list, 'classes': CLASSES}, MOTIF_CACHE)

sample = train_list[0]
print(f"Node feature shape: {sample.x.shape}  (should be [1024, 7])")

# ============================================================
# PRECOMPUTE k-NN GRAPHS (cached)
# ============================================================
def add_cached_graph(data, k=20):
    edge_index = knn_graph(data.pos, k=k, loop=False)
    data.edge_index = edge_index
    return data

print("\nPrecomputing k-NN graphs...")
train_list = [add_cached_graph(d, k=K_NEIGHBORS) for d in tqdm(train_list, desc='Train graphs')]
test_list = [add_cached_graph(d, k=K_NEIGHBORS) for d in tqdm(test_list, desc='Test graphs')]

# ============================================================
# STRATIFIED SUBSAMPLING
# ============================================================
def stratified_subsample(data_list, samples_per_class=100, seed=42):
    np.random.seed(seed)
    class_to_indices = {}
    for idx, data in enumerate(data_list):
        label = data.y.item() if isinstance(data.y, torch.Tensor) else data.y
        class_to_indices.setdefault(label, []).append(idx)

    keep_indices = []
    for label, indices in class_to_indices.items():
        if len(indices) > samples_per_class:
            keep = np.random.choice(indices, samples_per_class, replace=False)
        else:
            keep = indices
        keep_indices.extend(keep)

    return [data_list[i] for i in keep_indices]

TRAIN_SAMPLES_PER_CLASS = 100   # change as needed
USE_SUBSAMPLING = True

if USE_SUBSAMPLING:
    print(f"Subsampling training set to {TRAIN_SAMPLES_PER_CLASS} samples per class...")
    train_list_subsampled = stratified_subsample(train_list, samples_per_class=TRAIN_SAMPLES_PER_CLASS)
    print(f"Original train size: {len(train_list)}, after subsampling: {len(train_list_subsampled)}")
    train_loader = DataLoader(train_list_subsampled, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
else:
    train_loader = DataLoader(train_list, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

test_loader = DataLoader(test_list, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# ============================================================
# TRAINING FUNCTIONS (no mixed precision for simplicity)
# ============================================================
def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    for batch in tqdm(loader, desc='  train', leave=False):
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch)
        loss = F.cross_entropy(out, batch.y.squeeze())
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * batch.num_graphs
        correct += out.argmax(1).eq(batch.y.squeeze()).sum().item()
        total += batch.num_graphs
    return total_loss / total, correct / total

@torch.no_grad()
def test_epoch(model, loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    for batch in tqdm(loader, desc='  test ', leave=False):
        batch = batch.to(device)
        preds = model(batch).argmax(1).cpu().numpy()
        labels = batch.y.squeeze().cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels)
    return {
        'OA': accuracy_score(all_labels, all_preds) * 100,
        'mAcc': balanced_accuracy_score(all_labels, all_preds) * 100,
        'macro_f1': f1_score(all_labels, all_preds, average='macro', zero_division=0) * 100,
    }

# ============================================================
# BASELINE DGCNN MODEL DEFINITION
# ============================================================
class BaselineDGCNN(nn.Module):
    def __init__(self, in_channels=6, num_classes=40, k=20, dropout=0.5, small=False):
        super().__init__()
        self.k = k
        if small:
            ch1, ch2, ch3, ch4 = 32, 32, 64, 128
            embed_dims = 512
        else:
            ch1, ch2, ch3, ch4 = 64, 64, 128, 256
            embed_dims = 1024

        self.conv1 = EdgeConv(
            nn.Sequential(
                nn.Linear(2 * in_channels, ch1, bias=False),
                nn.BatchNorm1d(ch1),
                nn.LeakyReLU(0.2)
            ), aggr='max'
        )
        self.conv2 = EdgeConv(
            nn.Sequential(
                nn.Linear(2 * ch1, ch2, bias=False),
                nn.BatchNorm1d(ch2),
                nn.LeakyReLU(0.2)
            ), aggr='max'
        )
        self.conv3 = EdgeConv(
            nn.Sequential(
                nn.Linear(2 * ch2, ch3, bias=False),
                nn.BatchNorm1d(ch3),
                nn.LeakyReLU(0.2)
            ), aggr='max'
        )
        self.conv4 = EdgeConv(
            nn.Sequential(
                nn.Linear(2 * ch3, ch4, bias=False),
                nn.BatchNorm1d(ch4),
                nn.LeakyReLU(0.2)
            ), aggr='max'
        )

        total_channels = ch1 + ch2 + ch3 + ch4
        self.global_mlp = nn.Sequential(
            nn.Linear(2*total_channels, embed_dims, bias=False),
            nn.BatchNorm1d(embed_dims),
            nn.LeakyReLU(0.2),
            nn.Dropout(dropout)
        )
        self.classifier = nn.Sequential(
            nn.Linear(embed_dims, 512, bias=False),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(0.2),
            nn.Dropout(dropout),
            nn.Linear(512, 256, bias=False),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(0.2),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

    def forward(self, data):
        x = data.x[:, :6]          # pos + norm
        batch = data.batch
        pos = data.pos

        edge_index = knn_graph(pos, k=self.k, batch=batch, loop=False)
        x1 = self.conv1(x, edge_index)

        edge_index = knn_graph(x1, k=self.k, batch=batch, loop=False)
        x2 = self.conv2(x1, edge_index)

        edge_index = knn_graph(x2, k=self.k, batch=batch, loop=False)
        x3 = self.conv3(x2, edge_index)

        edge_index = knn_graph(x3, k=self.k, batch=batch, loop=False)
        x4 = self.conv4(x3, edge_index)

        x = torch.cat([x1, x2, x3, x4], dim=1)
        x_max = global_max_pool(x, batch)
        x_avg = global_mean_pool(x, batch)
        x = torch.cat([x_max, x_avg], dim=1)

        x = self.global_mlp(x)
        x = self.classifier(x)
        return x

# ============================================================
# TRAIN BASELINE MODEL
# ============================================================
SMALL_MODEL = True   # set False for full model

print("\n" + "="*80)
print("TRAINING BASELINE DGCNN")
print("="*80)

model = BaselineDGCNN(
    in_channels=6,
    num_classes=NUM_CLASSES,
    k=K_NEIGHBORS,
    dropout=0.5,
    small=SMALL_MODEL
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,} (small={SMALL_MODEL})")

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_oa = 0.0
patience = 20
patience_counter = 0

for epoch in trange(1, EPOCHS + 1, desc='Epochs'):
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, DEVICE)
    metrics = test_epoch(model, test_loader, DEVICE)
    scheduler.step()

    if metrics['OA'] > best_oa:
        best_oa = metrics['OA']
        patience_counter = 0
        torch.save(model.state_dict(), 'best_baseline.pt')
    else:
        patience_counter += 1

    if epoch % 10 == 0:
        print(f"\n  Epoch {epoch:3d} | loss {tr_loss:.4f} | trAcc {tr_acc*100:.1f}% | OA {metrics['OA']:.1f}% | mAcc {metrics['mAcc']:.1f}%")

    if patience_counter >= patience:
        print(f"Early stopping at epoch {epoch}")
        break

print("\n" + "="*80)
print(f"TRAINING COMPLETE")
print(f"Best OA: {best_oa:.2f}%")
print("="*80)

Loading data...
Loading motif-augmented cache …
Node feature shape: torch.Size([1024, 8])  (should be [1024, 7])

Precomputing k-NN graphs...


Train graphs:   0%|          | 0/9843 [00:00<?, ?it/s]

Test graphs:   0%|          | 0/2468 [00:00<?, ?it/s]

Subsampling training set to 100 samples per class...
Original train size: 9843, after subsampling: 3908

TRAINING BASELINE DGCNN
Model parameters: 691,624 (small=True)


Epochs:   0%|          | 0/150 [00:00<?, ?it/s]

  train:   0%|          | 0/489 [00:00<?, ?it/s]

  test :   0%|          | 0/309 [00:00<?, ?it/s]